# Comparative mid-training: 100M tokens per corpus

Based on the architecture in `v4.8_low_size_token.ipynb`. It trains four identical copies, initialized with the same seed, for 100M tokens on each corpus and logs *zero-shot* accuracy for HellaSwag, ARC-Easy, PIQA, and WinoGrande over 500 examples per benchmark in Weights & Biases.

> **Evaluation integrity.** Evaluation sets are never used as training data. Before running this notebook, confirm that the selected data (particularly Cosmopedia, which is synthetic) complies with the hackathon's current rules.

In [ ]:
%pip install -q wandb datasets sentencepiece huggingface_hub matplotlib

from __future__ import annotations

from collections.abc import Iterator
from pathlib import Path
import sys
import time
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{PROJECT_ROOT}[training,benchmarks]'])
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from torch.utils.data import DataLoader, IterableDataset

from llm_mini_lab.models import LoopedGPTModel
from llm_mini_lab.training import LOOPED_GPT_CONFIG, cosine_lr_multiplier, init_xavier, load_tokenizer, tokenizer_eot_id, tokenizer_vocab_size
from llm_mini_lab.evaluation import evaluate_benchmark_suite

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Project: {PROJECT_ROOT}\nDevice: {device}')

In [ ]:
# Shared configuration: preserves the v4.8 architecture.
SEED = 123
CONTEXT_LENGTH = 128
BATCH_SIZE = 8  # Reduce it if it does not fit in GPU memory.
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
TARGET_TOKENS_PER_DATASET = 100_000_000
BENCHMARK_MAX_EXAMPLES = 500
BENCHMARK_NAMES = ('hellaswag', 'arc_easy', 'piqa', 'winogrande')
WANDB_PROJECT = 'gpt2-50m-midtraining'

TOKENIZER_NAME = 'sp16384'
TOKENIZER_MODEL = PROJECT_ROOT / 'tokenizers' / 'fineweb_16384_bpe.model'
if TOKENIZER_NAME == 'sp16384' and not TOKENIZER_MODEL.is_file():
    raise FileNotFoundError(f'Tokenizer is missing: {TOKENIZER_MODEL}')
tokenizer = load_tokenizer(TOKENIZER_NAME, TOKENIZER_MODEL if TOKENIZER_NAME == 'sp16384' else None)
EOT_TOKEN_ID = tokenizer_eot_id(tokenizer)

MODEL_CONFIG = {
    **LOOPED_GPT_CONFIG,
    'context_length': CONTEXT_LENGTH,
    'vocab_size': tokenizer_vocab_size(tokenizer),
    'positional_encoding': 'rope',
    'tokenizer_name': TOKENIZER_NAME,
    'n_unique_layers': 3,
    'n_heads': 8,
    'emb_dim': 1024,
}

# Public dataset configurations. If a Hugging Face configuration changes, update only this dictionary.
DATASETS = {
    'fineweb_edu': {'path': 'HuggingFaceFW/fineweb-edu', 'name': 'sample-10BT', 'split': 'train', 'text_column': 'text'},
    'openwebmath': {'path': 'open-web-math/open-web-math', 'name': None, 'split': 'train', 'text_column': 'text'},
    'project_gutenberg': {'path': 'manu/project_gutenberg', 'name': None, 'split': 'en', 'text_column': 'text'},
    'cosmopedia': {'path': 'HuggingFaceTB/cosmopedia', 'name': 'web_samples_v2', 'split': 'train', 'text_column': 'text'},
}

def encode_text(text: str) -> list[int]:
    if hasattr(tokenizer, 'eot_token'):  # tiktoken
        return tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    return list(tokenizer.encode(text, out_type=int))  # SentencePiece

class StreamingTokenBlocks(IterableDataset):
    """Converts a streaming HF dataset into fixed-length next-token pairs."""
    def __init__(self, spec: dict, context_length: int):
        self.spec, self.context_length = spec, context_length

    def __iter__(self) -> Iterator[tuple[torch.Tensor, torch.Tensor]]:
        stream = load_dataset(self.spec['path'], self.spec['name'], split=self.spec['split'], streaming=True)
        stream = stream.shuffle(seed=SEED, buffer_size=10_000)
        buffer: list[int] = []
        for row in stream:
            text = row.get(self.spec['text_column'])
            if not isinstance(text, str) or not text.strip():
                continue
            buffer.extend(encode_text(text))
            buffer.append(EOT_TOKEN_ID)  # Never create artificial continuations across documents.
            while len(buffer) >= self.context_length + 1:
                block, buffer = buffer[: self.context_length + 1], buffer[self.context_length + 1 :]
                ids = torch.tensor(block, dtype=torch.long)
                yield ids[:-1], ids[1:]

def make_fresh_model() -> LoopedGPTModel:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    model = LoopedGPTModel(MODEL_CONFIG).to(device)
    model.apply(init_xavier)
    model.out_head.weight = model.tok_emb.weight
    n_params = sum(p.numel() for p in model.parameters())
    assert n_params <= 50_000_000, f'Model exceeds the parameter limit: {n_params:,}'
    return model

print(f'Model: {sum(p.numel() for p in make_fresh_model().parameters()):,} parameters')

In [ ]:
def train_and_evaluate(dataset_id: str, spec: dict) -> dict[str, float]:
    """Trains a fresh copy for ~100M tokens and logs its benchmarks to W&B."""
    model = make_fresh_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    updates = TARGET_TOKENS_PER_DATASET // (BATCH_SIZE * CONTEXT_LENGTH)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: cosine_lr_multiplier(step, max(1, int(0.05 * updates)), updates, 0.1),
    )
    loader = DataLoader(StreamingTokenBlocks(spec, CONTEXT_LENGTH), batch_size=BATCH_SIZE, num_workers=0)
    run = wandb.init(
        project=WANDB_PROJECT,
        name=f'midtraining-{dataset_id}-100m',
        group='midtraining-100m-comparison',
        job_type='midtraining',
        reinit='finish_previous',
        config={**MODEL_CONFIG, **spec, 'dataset_id': dataset_id, 'target_tokens': TARGET_TOKENS_PER_DATASET,
                'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE, 'benchmark_examples': BENCHMARK_MAX_EXAMPLES},
    )
    wandb.define_metric('update')
    wandb.define_metric('*', step_metric='update')
    model.train()
    tokens_seen, t0 = 0, time.perf_counter()
    for update, (inputs, targets) in enumerate(loader, start=1):
        if update > updates:
            break
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite loss in {dataset_id}, step {update}')
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        tokens_seen += inputs.numel()
        if update % 100 == 0 or update == updates:
            wandb.log({'update': update, 'tokens_seen': tokens_seen, 'train/loss': loss.item(),
                       'train/lr': optimizer.param_groups[0]['lr'], 'train/grad_norm': float(grad_norm)})
            print(f'{dataset_id} | {tokens_seen / 1e6:.1f}M tokens | loss={loss.item():.4f}')

    # One final point per benchmark and corpus; every metric uses 500 examples.
    results = evaluate_benchmark_suite(model, tokenizer, device, names=BENCHMARK_NAMES,
                                       max_examples=BENCHMARK_MAX_EXAMPLES, context_length=CONTEXT_LENGTH)
    missing_benchmarks = set(BENCHMARK_NAMES) - set(results)
    if missing_benchmarks:
        raise RuntimeError(f'Benchmark evaluation failed for: {sorted(missing_benchmarks)}')
    benchmark_metrics = {f'benchmark/{name}_accuracy': results[name]['accuracy'] for name in BENCHMARK_NAMES}
    wandb.log({'update': updates, 'tokens_seen': tokens_seen, **benchmark_metrics})

    checkpoint_dir = PROJECT_ROOT / 'checkpoints' / 'midtraining_100m'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / f'{dataset_id}_100m.pt'
    torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
                'config': MODEL_CONFIG, 'dataset': spec, 'tokens_seen': tokens_seen, 'benchmark_metrics': benchmark_metrics}, checkpoint_path)
    wandb.save(str(checkpoint_path))
    wandb.summary.update({'elapsed_seconds': time.perf_counter() - t0, **benchmark_metrics})
    run.finish()
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'dataset': dataset_id, **{name: values['accuracy'] for name, values in results.items()}}

comparison = [train_and_evaluate(dataset_id, spec) for dataset_id, spec in DATASETS.items()]

In [ ]:
# Comparative chart: logged to W&B as an image and a table.
run = wandb.init(project=WANDB_PROJECT, name='midtraining-100m-summary', group='midtraining-100m-comparison', job_type='summary', reinit='finish_previous')
table = wandb.Table(columns=['dataset', *BENCHMARK_NAMES])
for row in comparison:
    table.add_data(row['dataset'], *[row[name] for name in BENCHMARK_NAMES])

fig, ax = plt.subplots(figsize=(11, 5))
for name in BENCHMARK_NAMES:
    ax.plot([row['dataset'] for row in comparison], [row[name] for row in comparison], marker='o', linewidth=2, label=name)
ax.set(title='Zero-shot accuracy after 100M tokens per corpus', xlabel='Mid-training corpus', ylabel='Accuracy')
ax.set_ylim(0, 1); ax.grid(alpha=0.25); ax.legend(ncol=2)
plt.xticks(rotation=15); plt.tight_layout()
wandb.log({'benchmark_comparison': table, 'benchmark_accuracy_by_dataset': wandb.Image(fig)})
plt.show()
run.finish()